# Build artist_points_flat.json

In [32]:
import json
from itertools import groupby

In [39]:
with open("../../data/colab/ar_clusters.json", encoding="utf-8") as f: artist_data = json.load(f)
with open("../../data/colab/inst_clusters.json", encoding="utf-8") as f: institution_data = json.load(f)

In [40]:
def build_flat_from_source(data: list, source_label: str, k_to_n: dict) -> dict:
    points = {}

    for n_entry in data:
        k = n_entry["n"]
        n = k_to_n.get(k)
        if n is None:
            continue
        for cluster in n_entry["clusters"]:
            cluster_key = int(cluster["key"])
            for point in cluster["data"]:
                source_idx = point.get("source_idx")
                artist_id  = point.get("artist_id")
                text       = point.get("text")
                if not source_idx or artist_id is None:
                    continue

                key = (source_idx, text)

                if key not in points:
                    points[key] = {
                        "id":          point.get("point"),
                        "artist_id":      artist_id,
                        "text":           text,
                        "is_artwork":     point.get("is_artwork"),
                        "institution_id": point.get("institution_id"),
                        "source_actor":   source_label,
                        "cluster_by_n":   {"artist": {}, "institution": {}},
                    }

                points[key]["cluster_by_n"][source_label][n] = cluster_key

    return points

In [41]:
with open("../../data/clusters/artist_cluster_summary.json") as f:
    ar_k_to_n = {e["k"]: e["n"] for e in json.load(f)["numClusters"]}
with open("../../data/clusters/institution_cluster_summary.json") as f:
    inst_k_to_n = {e["k"]: e["n"] for e in json.load(f)["numClusters"]}

artist_points_raw      = build_flat_from_source(artist_data,      source_label="artist",      k_to_n=ar_k_to_n)
institution_points_raw = build_flat_from_source(institution_data, source_label="institution", k_to_n=inst_k_to_n)

all_points = dict(artist_points_raw)
for key, point in institution_points_raw.items():
    if key in all_points:
        all_points[key]["cluster_by_n"]["institution"] = point["cluster_by_n"]["institution"]
    else:
        all_points[key] = point

sorted_points = sorted(all_points.items(), key=lambda x: (x[1]["artist_id"], x[0]))

artist_grouped = [
    {
        "artist_id": artist_id,
        "points": [
            {"source_idx": key[0], **{k: v for k, v in p.items() if k != "artist_id"}}
            for key, p in points
        ]
    }
    for artist_id, points in groupby(sorted_points, key=lambda x: x[1]["artist_id"])
]

with open("../../data/clusters/artist_points_flat.json", "w", encoding="utf-8") as f:
    json.dump(artist_grouped, f, indent=2, ensure_ascii=False)

print(f"{len(artist_grouped)} artists, {len(all_points)} unique points")

30 artists, 964 unique points
